# 09 — MQTT ingestion + SDK read-back

MQTT in Cenotoo is an **ingestion** transport. The broker + bridge route every
valid publish into Kafka, and the consumer persists it to Cassandra.

```
device --MQTT publish--> Mosquitto --> mqtt-bridge --> Kafka --> consumer --> Cassandra
                                                                              |
                                                          SDK GET /get_data --+
```

This notebook keeps the MQTT side intentionally small:

1. Provision a project/collection and a **write** key with the SDK
2. Publish one reading, then a small batch, over MQTT with QoS 1
3. Read the data back and aggregate it with the SDK (REST)

**Topic contract:** exactly three slash-separated segments — `{org}/{project}/{collection}`.
Anything else is dropped by the bridge.

**Auth contract:** `username = <project_id>` (UUID) and `password = <write-or-master API key>`.
Production ACLs allow **publish only**; reading is done through REST or Kafka, not MQTT subscribe.

Prerequisites:

```bash
pip install -r examples/notebooks/requirements.txt
cp examples/notebooks/env.example .env
```

This notebook targets the **currently exposed TLS interfaces**:

| Service | URL |
|---|---|
| REST API (read-back) | `https://api.cenotoo.local` |
| MQTT | `mqtts://mqtt.cenotoo.local:8883` |
| Trust | `cenotoo-local-ca.crt` (self-signed local CA) |

`MQTT_HOST`, `MQTT_PORT` and `MQTT_USE_TLS` default to the TLS listener. For a
local plaintext broker only, override `MQTT_HOST=localhost MQTT_PORT=1883 MQTT_USE_TLS=false`.

In [ ]:
%pip install -q "nostradamus-ioto-sdk @ git+https://github.com/robotics-4-all/nostradamus-ioto-sdk.git@4f8e7fc7c28c859eefed9f7382d1b148a5c6eeb8" "paho-mqtt==2.1.0" python-dotenv

## 1. Configuration and helpers

In [ ]:
import json
import os
import random
import time
from datetime import datetime, timedelta, timezone

from dotenv import load_dotenv
from nostradamus_ioto_sdk import NostradamusClient
from nostradamus_ioto_sdk.models import KeyType

load_dotenv()

BASE_URL = os.getenv("CENOTOO_BASE_URL", "https://api.cenotoo.local")
USERNAME = os.getenv("CENOTOO_USERNAME", "")
PASSWORD = os.getenv("CENOTOO_PASSWORD", "")
MQTT_HOST = os.getenv("MQTT_HOST", "mqtt.cenotoo.local")
MQTT_PORT = int(os.getenv("MQTT_PORT", "8883"))
MQTT_USE_TLS = os.getenv("MQTT_USE_TLS", "true").lower() == "true"

def resolve_ca_bundle():
    configured = os.getenv("MQTT_CA_CERT") or os.getenv("CENOTOO_CA_BUNDLE", "")
    for candidate in (configured, "cenotoo-local-ca.crt", "../../cenotoo-local-ca.crt"):
        if candidate and os.path.isfile(os.path.abspath(os.path.expanduser(candidate))):
            return os.path.abspath(os.path.expanduser(candidate))
    return None


MQTT_CA_CERT = resolve_ca_bundle()
if MQTT_CA_CERT:
    os.environ["SSL_CERT_FILE"] = MQTT_CA_CERT

assert USERNAME and PASSWORD, "SKIP: set CENOTOO_USERNAME and CENOTOO_PASSWORD (see env.example)"


def unique_name(prefix):
    return f"{prefix}_{time.strftime('%Y%m%d%H%M%S')}_{random.randint(1000, 9999)}"


def wait_for(fn, predicate, timeout=120, interval=3, description="condition"):
    deadline = time.monotonic() + timeout
    last = None
    while time.monotonic() < deadline:
        last = fn()
        if predicate(last):
            return last
        time.sleep(interval)
    raise TimeoutError(f"timed out waiting for {description}; last={last!r}")


def show(rows, title="", limit=20):
    if title:
        print(f"\n{title}")
    if isinstance(rows, dict):
        rows = [rows]
    rows = list(rows or [])[:limit]
    if not rows:
        print("  (no rows)")
        return
    columns = list(rows[0].keys())
    print("  " + " | ".join(columns))
    for row in rows:
        print("  " + " | ".join(str(row.get(column)) for column in columns))


## 2. Provision the destination with the SDK

The MQTT topic is built from the **real** organization/project/collection names,
because the bridge maps `{org}/{project}/{collection}` to the Kafka topic
`{org}.{project}.{collection}`.

In [ ]:
TLS_KWARGS = {"verify": MQTT_CA_CERT} if (MQTT_CA_CERT and ".local" in BASE_URL) else {}
admin = NostradamusClient(base_url=BASE_URL, username=USERNAME, password=PASSWORD, **TLS_KWARGS)
organization = admin.organizations.get()

project = admin.projects.create(
    name=unique_name("mqtt_demo"),
    description="MQTT simple ingestion example",
    tags=["notebook", "mqtt"],
)

collection = admin.collections.create(
    project_id=project.project_id,
    name=unique_name("greenhouse"),
    description="Greenhouse sensor readings",
    collection_schema={
        "temperature": 0.0,
        "humidity": 0,
        "soil_moisture": 0.0,
        "room": "north",
    },
    tags=["notebook", "mqtt"],
)

write_key = admin.project_keys.create(project.project_id, KeyType.WRITE)
read_key = admin.project_keys.create(project.project_id, KeyType.READ)
read_client = NostradamusClient(base_url=BASE_URL, api_key=read_key.api_key, **TLS_KWARGS)

TOPIC = f"{organization.organization_name}/{project.project_name}/{collection.collection_name}"
DEVICE_KEY = "greenhouse-01"

print("organization :", organization.organization_name)
print("project      :", project.project_name, project.project_id)
print("collection   :", collection.collection_name, collection.collection_id)
print("MQTT topic   :", TOPIC)

## 3. Connect to the broker

`username` is the **project UUID**, `password` is the **write API key**.
Both are needed when the Mosquitto go-auth plugin is enabled (the production default).

In [ ]:
import paho.mqtt.client as mqtt


def connect_mqtt():
    client = mqtt.Client(
        callback_api_version=mqtt.CallbackAPIVersion.VERSION2,
        client_id=f"cenotoo-nb-{random.randint(1000, 9999)}",
    )
    client.username_pw_set(str(project.project_id), write_key.api_key)
    if MQTT_USE_TLS:
        client.tls_set(ca_certs=MQTT_CA_CERT)
    client.connect(MQTT_HOST, MQTT_PORT, keepalive=60)
    client.loop_start()
    time.sleep(0.5)
    return client


client = connect_mqtt()
print(f"connected to {'mqtts' if MQTT_USE_TLS else 'mqtt'}://{MQTT_HOST}:{MQTT_PORT}")

## 4. Publish one reading

The payload is ordinary JSON. `key` is required (it is the Cassandra partition key);
`timestamp` is optional, but sending it explicitly makes ordering deterministic.
The bridge/Kafka consumer fills in `timestamp`, `day` and `id` when they are missing.

In [ ]:
def publish(payload, qos=1):
    info = client.publish(TOPIC, json.dumps(payload), qos=qos)
    info.wait_for_publish(timeout=10)
    if info.rc != mqtt.MQTT_ERR_SUCCESS:
        raise RuntimeError(f"publish failed: rc={info.rc}")
    return info


now = datetime.now(timezone.utc).replace(microsecond=0)
publish(
    {
        "key": DEVICE_KEY,
        "timestamp": now.isoformat().replace("+00:00", "Z"),
        "temperature": 24.5,
        "humidity": 58,
        "soil_moisture": 37.2,
        "room": "north",
    }
)
print("published one reading with QoS 1")

## 5. Publish a small batch

Reuse one connection and publish five readings one minute apart.
This is the whole "fleet" pattern: a loop, a JSON payload, QoS 1.

In [ ]:
base_time = now - timedelta(minutes=4)
for step in range(5):
    publish(
        {
            "key": DEVICE_KEY,
            "timestamp": (base_time + timedelta(minutes=step))
            .isoformat()
            .replace("+00:00", "Z"),
            "temperature": round(22.0 + 0.9 * step, 2),
            "humidity": 52 + step,
            "soil_moisture": round(34.0 + 1.1 * step, 2),
            "room": "north",
        }
    )
print("published 5 readings")

## 6. Read back with the SDK

MQTT has no read path for project keys in production (publish-only ACL), so the
consumer side uses the REST API through the SDK: `GET /get_data`.

The write path is asynchronous (MQTT → bridge → Kafka → consumer → Cassandra),
so the notebook polls until all readings are visible.

In [ ]:
def rows_for_device(limit=20):
    return read_client.data.get(
        project.project_id,
        collection.collection_id,
        attributes=["timestamp", "temperature", "humidity", "soil_moisture", "room"],
        filters=[
            {"property_name": "key", "operator": "eq", "property_value": DEVICE_KEY}
        ],
        order_by=json.dumps({"field": "timestamp", "order": "asc"}),
        limit=limit,
    )


result = wait_for(
    lambda: rows_for_device(),
    lambda response: len(response.data) >= 6,
    description="MQTT readings to land in Cassandra",
)
show(result.data, "MQTT → Kafka → Cassandra → SDK read-back")

## 7. Aggregate the ingested readings with the SDK

The MQTT side stays simple; aggregation is the platform's `GET /statistics`
endpoint, reached through the SDK. Readings are one minute apart, so
`every_1_minutes` produces one bucket per reading.

In [ ]:
avg = read_client.data.statistics(
    project.project_id,
    collection.collection_id,
    stat="avg",
    attribute="temperature",
    interval="every_1_minutes",
)
show(avg, "Average temperature per 1-minute bucket")

count = read_client.data.statistics(
    project.project_id,
    collection.collection_id,
    stat="count",
    attribute="temperature",
    interval="every_1_minutes",
)
show(count, "Reading count per 1-minute bucket")

> **MQTT notes**
> - A topic without exactly three non-empty segments is dropped by the bridge — publish
>   success from the broker does not guarantee persistence.
> - Use `qos=1` for at-least-once delivery from device to broker; the bridge/Kafka path
>   handles the rest.
> - For high-rate streams, keep one connection and call `publish()` in a loop rather than
>   reconnecting per message.

## 8. Clean up

In [ ]:
import contextlib

with contextlib.suppress(Exception):
    client.loop_stop()
    client.disconnect()

with contextlib.suppress(Exception):
    read_client.close()

with contextlib.suppress(Exception):
    admin.project_keys.delete(project.project_id, read_key.api_key)
with contextlib.suppress(Exception):
    admin.project_keys.delete(project.project_id, write_key.api_key)
with contextlib.suppress(Exception):
    admin.collections.delete(project.project_id, collection.collection_id)
with contextlib.suppress(Exception):
    admin.projects.delete(project.project_id)
with contextlib.suppress(Exception):
    admin.close()

print("cleanup done")